# MLB Hit Probability: Random Forest vs. Neural Network

Recreation of the @flanalytics project: predict whether a batted ball becomes a hit using 2026 Statcast data,
comparing a random forest against a small neural network, under two missing-data strategies.

**Where this differs from the video (and why)**

1. **Apples-to-apples metrics.** The video's random forest is a `RandomForestRegressor` and its "67% accuracy" is actually
   an R² score, while the neural net reports classification accuracy. Those aren't comparable. This notebook keeps the
   original regressor for reference, adds a `RandomForestClassifier`, and scores every model with the same metrics
   (accuracy, ROC AUC, log loss, Brier score) on the **same** train/test split.
2. **A "dumb" baseline.** Roughly two-thirds of balls in play are outs, so always guessing "out" already scores high
   accuracy. That number is printed so model accuracy has context.
3. **Regular season only + explicit bunt removal.** The date range can pull in spring-training games, and the
   `events` filter alone doesn't remove bunt singles or bunt groundouts.
4. **Leakage check.** `hit_location` is the fielder who first touched the ball, which is recorded *after* the outcome.
   It also tends to be empty for home runs, so the strict `dropna` quietly removes them. A diagnostic cell shows this,
   and the final section retrains on a leak-free feature set.
5. **Imputation fit on training data only**, so the test set doesn't leak into the medians/modes.

## 1. Setup

In [ ]:
# Python 3.11+ (the video used 3.11.1)
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import pybaseball
from pybaseball import statcast, cache

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    r2_score, mean_squared_error, accuracy_score,
    roc_auc_score, log_loss, brier_score_loss,
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

cache.enable()
RANDOM_STATE = 42
keras.utils.set_random_seed(RANDOM_STATE)  # seeds Python, NumPy and TF for reproducible NN runs

In [ ]:
print(f'Pybaseball Version: {pybaseball.__version__}')
print(f'Pandas Version: {pd.__version__}')
print(f'NumPy Version: {np.__version__}')
print(f'Matplotlib Version: {matplotlib.__version__}')
print(f'Scikit Version: {sklearn.__version__}')
print(f'Tensorflow Version: {tf.__version__}')

## 2. Pull the 2026 season from Statcast

A full season is ~780k pitches and takes a while to download, so the raw pull is saved to disk the first time
and reloaded on later runs. Delete the `.pkl` file to force a fresh pull.

In [ ]:
RAW_PATH = Path("statcast_2026.pkl")

if RAW_PATH.exists():
    data = pd.read_pickle(RAW_PATH)
else:
    # Start and end dates for 2026 MLB regular season (Opening Night -> final day)
    data = statcast(start_dt="2026-03-25", end_dt="2026-09-27")
    data.to_pickle(RAW_PATH)

print(f"Raw pitches: {data.shape}")
print(data["game_type"].value_counts())

## 3. Keep regular-season, non-bunt batted balls and build the target

`hit` = 1 for single/double/triple/home run, 0 for everything else in the list (errors count as non-hits, as in
official scoring).

In [ ]:
events_to_keep = [
    'field_out', 'force_out', 'single', 'double', 'triple', 'home_run',
    'field_error', 'grounded_into_double_play', 'sac_fly', 'fielders_choice',
    'double_play', 'fielders_choice_out', 'sac_fly_double_play', 'triple_play',
]
hit_events = ['single', 'double', 'triple', 'home_run']

bip = data[(data["game_type"] == "R") & (data["events"].isin(events_to_keep))]

# The events list drops sac bunts, but bunt singles / bunt groundouts slip through; catch them via the play text
is_bunt = bip["des"].str.contains(r"\bbunt", case=False, na=False, regex=True)
print(f"Bunts removed: {is_bunt.sum():,}")
bip = bip[~is_bunt].copy()

bip["hit"] = bip["events"].isin(hit_events).astype(int)

print(f"Batted balls: {bip.shape}")
print(f"Hit rate: {bip['hit'].mean():.3f}")

## 4. Feature set + what the strict `dropna` actually throws away

In [ ]:
features = [
    "launch_speed",
    "launch_angle",
    "pitch_type",
    "hit_location",
    "bat_speed",
    "swing_length",
    "swing_path_tilt",
    "if_fielding_alignment",
    "of_fielding_alignment",
    "hit_distance_sc",
    "release_speed",
]
categorical = ["pitch_type", "if_fielding_alignment", "of_fielding_alignment"]
numeric = [c for c in features if c not in categorical]

print("Share of rows missing each feature:")
print(bip[features].isna().mean().sort_values(ascending=False).round(3))

In [ ]:
# How each outcome survives the strict "complete rows only" filter.
# Watch the home_run row: if most home runs vanish, the model never learns what a homer looks like.
strict_rows = bip.dropna(subset=features)
survival = pd.DataFrame({
    "before": bip["events"].value_counts(),
    "after_strict_dropna": strict_rows["events"].value_counts(),
}).fillna(0).astype(int)
survival["kept_pct"] = (100 * survival["after_strict_dropna"] / survival["before"]).round(1)
survival.sort_values("before", ascending=False)

## 5. Shared helpers

Both missing-data strategies and both model types go through the same functions, so every row of the final
comparison table is produced the same way.

In [ ]:
def prepare_strict(df, features, categorical):
    # Method A (video's approach): only keep batted balls with every feature present.
    d = df.dropna(subset=features)
    X = pd.get_dummies(d[features], columns=categorical, dtype=float)
    y = d["hit"].to_numpy()
    return train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)


def prepare_imputed(df, features, categorical):
    # Method B: fill numeric gaps with the median and categorical gaps with the most common value.
    # Fill values are learned from the training split only, then applied to both splits.
    X_raw, y = df[features], df["hit"].to_numpy()
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    num = [c for c in features if c not in categorical]
    fill = pd.concat([X_tr[num].median(), X_tr[categorical].mode().iloc[0]])
    X_tr, X_te = X_tr.fillna(fill), X_te.fillna(fill)

    X_tr = pd.get_dummies(X_tr, columns=categorical, dtype=float)
    # A rare pitch type might only appear in one split; align test columns to train
    X_te = pd.get_dummies(X_te, columns=categorical, dtype=float).reindex(columns=X_tr.columns, fill_value=0.0)
    return X_tr, X_te, y_tr, y_te


def score(name, method, y_true, p_hit, train_time):
    p = np.clip(p_hit, 1e-7, 1 - 1e-7)
    return {
        "model": name,
        "imputation": method,
        "accuracy": accuracy_score(y_true, p >= 0.5),
        "roc_auc": roc_auc_score(y_true, p),
        "log_loss": log_loss(y_true, p, labels=[0, 1]),
        "brier": brier_score_loss(y_true, p),
        "train_time_s": train_time,
    }


def run_rf_classifier(X_tr, X_te, y_tr, y_te, method):
    start = time.time()
    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    t = time.time() - start
    return rf, score("Random forest", method, y_te, rf.predict_proba(X_te)[:, 1], t)


def build_nn(n_features):
    # Input -> one hidden layer of 64 ReLU units -> 2-unit softmax (P(out), P(hit)), as in the video
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(2, activation="softmax"),
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def run_nn(X_tr, X_te, y_tr, y_te, method, verbose=0):
    scaler = StandardScaler()  # NNs train poorly on unscaled inputs; fit on train only
    X_tr_s = np.asarray(scaler.fit_transform(X_tr), dtype=np.float32)
    X_te_s = np.asarray(scaler.transform(X_te), dtype=np.float32)
    y_tr_i = np.asarray(y_tr, dtype=np.int32)

    model = build_nn(X_tr_s.shape[1])
    start = time.time()
    history = model.fit(
        X_tr_s, y_tr_i, epochs=20, batch_size=100, validation_split=0.2, verbose=verbose
    )
    t = time.time() - start
    p_hit = model.predict(X_te_s, verbose=0)[:, 1]
    return model, history, score("Neural network", method, y_te, p_hit, t)

## 6. Method A: strict (complete rows only), same split for every model

In [ ]:
X_train, X_test, y_train, y_test = prepare_strict(bip, features, categorical)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Encoded features ({X_train.shape[1]}):")
print(list(X_train.columns))

majority_acc = 1 - y_test.mean()
print(f"\nAlways-predict-'out' accuracy: {majority_acc:.4f}  <- every model has to beat this")

### 6a. The video's random forest (regressor), kept for reference

This predicts a number between 0 and 1 and is scored with R² and RMSE. **R² is not accuracy**, which is why the
video's 67% vs. 90% comparison is misleading.

In [ ]:
start_time = time.time()
baseline_rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
baseline_rf.fit(X_train, y_train)
baseline_time = time.time() - start_time

baseline_predictions = baseline_rf.predict(X_test)
print(f'Baseline RF R^2 Score: {r2_score(y_test, baseline_predictions):.4f}')
print(f'Baseline RF RMSE: {np.sqrt(mean_squared_error(y_test, baseline_predictions)):.4f}')
print(f'Baseline RF Training Time: {baseline_time:.2f}s')

# Its output can be read as a hit probability, so it can be scored like a classifier too:
print(f"Same regressor scored as a classifier -> accuracy: "
      f"{accuracy_score(y_test, baseline_predictions >= 0.5):.4f}")

### 6b. Random forest classifier (the fair comparison)

In [ ]:
rf_strict, rf_strict_scores = run_rf_classifier(X_train, X_test, y_train, y_test, "Strict (dropna)")
pd.Series(rf_strict_scores)

### 6c. Feature importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Impurity-based importance (what the video showed): fast, but biased toward high-cardinality numeric features
imp = pd.Series(rf_strict.feature_importances_, index=X_train.columns).sort_values().tail(15)
imp.plot.barh(ax=axes[0], title="Impurity importance (top 15)")

# Permutation importance: how much accuracy drops when a feature is shuffled. More trustworthy.
rng = np.random.default_rng(RANDOM_STATE)
pos = rng.choice(len(X_test), size=min(5000, len(X_test)), replace=False)  # subsample: full test set is slow
perm = permutation_importance(
    rf_strict, X_test.iloc[pos], y_test[pos],
    n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
perm_imp = pd.Series(perm.importances_mean, index=X_train.columns).sort_values().tail(15)
perm_imp.plot.barh(ax=axes[1], title="Permutation importance (top 15)")

plt.tight_layout()
plt.show()

### 6d. Neural network

In [ ]:
build_nn(X_train.shape[1]).summary()

nn_strict, nn_strict_history, nn_strict_scores = run_nn(
    X_train, X_test, y_train, y_test, "Strict (dropna)", verbose=1
)
pd.Series(nn_strict_scores)

In [ ]:
h = nn_strict_history.history
plt.figure(figsize=(8, 4))
plt.plot(h["accuracy"], label="Training accuracy")
plt.plot(h["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Neural network: training vs. validation")
plt.legend()
plt.show()

## 7. Method B: median / most-common imputation

In [ ]:
X_train_b, X_test_b, y_train_b, y_test_b = prepare_imputed(bip, features, categorical)
print(f"Train: {X_train_b.shape}   Test: {X_test_b.shape}  (more rows than Method A, nothing dropped)")

_, rf_imp_scores = run_rf_classifier(X_train_b, X_test_b, y_train_b, y_test_b, "Median / mode")
_, _, nn_imp_scores = run_nn(X_train_b, X_test_b, y_train_b, y_test_b, "Median / mode")

## 8. Model comparison table

In [ ]:
comparison = pd.DataFrame([rf_strict_scores, nn_strict_scores, rf_imp_scores, nn_imp_scores])
comparison.round(4)

## 9. Going further: a leak-free feature set

`hit_location` (who fielded it) and `hit_distance_sc` (measured flight distance) are partly recorded after the ball
is in play, so they hand the model hints about the outcome. Here they're swapped for **spray angle**, the ball's
horizontal direction off the bat computed from `hc_x`/`hc_y`, flipped by batter handedness so positive always
means "pulled." This is closer to how public expected-batting-average models are built.

Compare these scores to Section 8. A drop is expected; what remains is how well the *contact itself* predicts a hit.

In [ ]:
spray_raw = np.degrees(np.arctan2(bip["hc_x"] - 125.42, 198.27 - bip["hc_y"]))
bip["spray_angle_pull"] = np.where(bip["stand"] == "R", -spray_raw, spray_raw)

leak_free_features = [
    "launch_speed", "launch_angle", "spray_angle_pull",
    "bat_speed", "swing_length", "swing_path_tilt", "release_speed",
    "pitch_type", "if_fielding_alignment", "of_fielding_alignment",
]

X_tr_lf, X_te_lf, y_tr_lf, y_te_lf = prepare_strict(bip, leak_free_features, categorical)
print(f"Leak-free rows: {len(X_tr_lf) + len(X_te_lf):,}")

_, rf_lf = run_rf_classifier(X_tr_lf, X_te_lf, y_tr_lf, y_te_lf, "Strict, leak-free")
_, _, nn_lf = run_nn(X_tr_lf, X_te_lf, y_tr_lf, y_te_lf, "Strict, leak-free")

full_comparison = pd.DataFrame(
    [rf_strict_scores, nn_strict_scores, rf_imp_scores, nn_imp_scores, rf_lf, nn_lf]
)
full_comparison.round(4)